In [ ]:
# Preview which empty non-cartoon Silver bodies the updated parser can recover.
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

table_name = '`news-analytics`.silver.articles'
silver = spark.table(table_name)
to_repair = (silver
    .filter(F.trim(F.col('body_text')) == '')
    .filter(~F.coalesce(F.array_contains(F.col('categories'), 'Cartoons'), F.lit(False)))
    .select('canonical_url', 'source_path'))
before_count = silver.count()
before_empty = to_repair.count()
paths = [row.source_path for row in to_repair.select('source_path').distinct().collect()]
if not paths:
    raise ValueError('No empty non-cartoon article bodies to repair')
print('ARTICLE COUNT BEFORE:', before_count)
print('EMPTY NON-CARTOON BODIES BEFORE:', before_empty)

# Parse only those archived files and keep rows with recovered article text.
repair_schema = StructType([
    StructField('canonical_url', StringType()),
    StructField('source_path', StringType()),
    StructField('body_text', StringType()),
    StructField('links', silver.schema['links'].dataType),
])
def parse_file(path, content):
    from parse_article import parse_article
    article = parse_article(content.decode('utf-8', errors='replace'))
    if article is None:
        return None
    return {
        'canonical_url': article['canonical_url'],
        'source_path': path,
        'body_text': article['body_text'],
        'links': article['links'],
    }

source_files = spark.read.format('binaryFile').load(paths).select('path', 'content')
parsed = (source_files
    .select(F.udf(parse_file, repair_schema)('path', 'content').alias('article'))
    .filter(F.col('article').isNotNull())
    .select('article.*'))
recoverable = parsed.filter(F.trim(F.col('body_text')) != '').cache()
recovered_count = recoverable.count()
print('RECOVERABLE BODIES:', recovered_count)
display(recoverable.select(
    'canonical_url', 'source_path', F.length('body_text').alias('body_chars')
).orderBy('canonical_url').limit(20))

# Show files that still have no recoverable body or no matching canonical URL.
unrecovered = to_repair.join(
    recoverable.select('canonical_url', 'source_path'),
    ['canonical_url', 'source_path'],
    'left_anti')
print('NOT RECOVERED:', unrecovered.count())
display(unrecovered.orderBy('canonical_url').limit(20))


In [ ]:
# Update only matching Silver rows whose body is still empty.
recoverable.createOrReplaceTempView('recoverable_articles')
merge_result = spark.sql(f'''
    MERGE INTO {table_name} AS target
    USING recoverable_articles AS source
    ON target.canonical_url = source.canonical_url
        AND target.source_path = source.source_path
    WHEN MATCHED AND TRIM(target.body_text) = '' AND TRIM(source.body_text) <> ''
        THEN UPDATE SET body_text = source.body_text, links = source.links
''')
print('MERGE RESULT:', merge_result.collect())


In [ ]:
# Verify that article count is unchanged and see which non-cartoon bodies remain empty.
silver_after = spark.table(table_name)
remaining = (silver_after
    .filter(F.trim(F.col('body_text')) == '')
    .filter(~F.coalesce(F.array_contains(F.col('categories'), 'Cartoons'), F.lit(False))))
print('ARTICLE COUNT:', before_count, '->', silver_after.count())
print('EMPTY NON-CARTOON BODIES:', before_empty, '->', remaining.count())
display(remaining.select('canonical_url', 'source_path', 'categories')
    .orderBy('canonical_url').limit(20))
recoverable.unpersist()
